In [14]:

%pip install torch transformers shap pandas numpy matplotlib


In [15]:

from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:

from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = Path(
    "/content/drive/MyDrive/NewsBias/models/newsbias_roberta_best"
)

print("Model folder exists:", MODEL_DIR.exists())

if not MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Folder not found: {MODEL_DIR}\n"
        "Check the folder location in Google Drive."
    )

print("Files:", [p.name for p in MODEL_DIR.iterdir()])

assert (MODEL_DIR / "config.json").exists(), "config.json is missing"
assert (
    (MODEL_DIR / "model.safetensors").exists()
    or (MODEL_DIR / "pytorch_model.bin").exists()
), "Model weights are missing"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR)
).to(device)

model.eval()

print("\nDevice:", device)
print("Label mapping:", model.config.id2label)
print("Model loaded successfully!")


Model folder exists: True
Files: ['config.json', 'training_args.bin', 'tokenizer.json', 'tokenizer_config.json', 'model.safetensors']


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Device: cuda
Label mapping: {0: 'Non-biased', 1: 'Biased'}
Model loaded successfully!


In [17]:

import torch

test_text = (
    "The government announced a new policy today, "
    "according to officials."
)

inputs = tokenizer(
    test_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(device)

with torch.no_grad():
    outputs = model(**inputs)
    probabilities = torch.softmax(outputs.logits, dim=-1)[0]

predicted_id = int(torch.argmax(probabilities).item())
label = model.config.id2label[predicted_id]

print("Text:", test_text)
print("Predicted label:", label)
print("Probabilities:")

for i, probability in enumerate(probabilities.tolist()):
    print(model.config.id2label[i], f"{probability:.4f}")


Text: The government announced a new policy today, according to officials.
Predicted label: Non-biased
Probabilities:
Non-biased 0.9939
Biased 0.0061


In [18]:

%pip install -q shap


In [19]:

import numpy as np
import shap
import torch

# Identify the output index for the Biased class
id2label = model.config.id2label

biased_indices = [
    int(idx) for idx, name in id2label.items()
    if "biased" in str(name).lower()
    and "non" not in str(name).lower()
]

if len(biased_indices) != 1:
    raise ValueError(f"Could not identify Biased class: {id2label}")

biased_idx = biased_indices[0]
print("Explaining class:", id2label[biased_idx])
print("Class index:", biased_idx)

# Return the model's probability for the Biased class
def predict_biased_probability(texts):
    if isinstance(texts, str):
        texts = [texts]

    results = []

    for start in range(0, len(texts), 8):
        batch = list(texts[start:start + 8])

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1)[:, biased_idx]

        results.extend(probs.detach().cpu().numpy().tolist())

    return np.array(results)

# Start with the same test sentence
explanation_text = (
    "The government announced a new policy today, "
    "according to officials."
)

masker = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(
    predict_biased_probability,
    masker
)

shap_values = explainer(
    [explanation_text],
    max_evals=100
)

print("Original Biased probability:",
      predict_biased_probability([explanation_text])[0])

shap.plots.text(shap_values[0])


Explaining class: Biased
Class index: 1
Original Biased probability: 0.006086498498916626


In [20]:

second_text = (
    "The corrupt government shamelessly betrayed ordinary citizens "
    "with its disastrous policy."
)

print("Text:", second_text)
print("Biased probability:",
      predict_biased_probability([second_text])[0])

second_values = explainer([second_text], max_evals=100)
shap.plots.text(second_values[0])


Text: The corrupt government shamelessly betrayed ordinary citizens with its disastrous policy.
Biased probability: 0.9970069527626038




import pandas as pd
import numpy as np

def get_word_level_importance(shap_values, top_n=10):
    explanation = shap_values[0]
    segments = list(explanation.data)
    contributions = np.asarray(explanation.values).reshape(-1)

    words = []

    for segment, score in zip(segments, contributions):
        segment = str(segment)

        # Ignore empty boundary segments
        if not segment.strip():
            continue

        # SHAP text segments include spaces where a new word begins
        starts_new_word = segment[0].isspace()
        piece = segment.strip()

        if starts_new_word or not words:
            words.append({
                "token": piece,
                "shap_value": float(score)
            })
        else:
            # Merge a subword fragment into the preceding word
            words[-1]["token"] += piece
            words[-1]["shap_value"] += float(score)

    df = pd.DataFrame(words)

    if df.empty:
        return df

    df["direction"] = np.where(
        df["shap_value"] > 0,
        "Pushes toward Biased",
        "Pushes away from Biased"
    )

    df["absolute_importance"] = df["shap_value"].abs()

    return (
        df.sort_values("absolute_importance", ascending=False)
          .head(top_n)
          .drop(columns="absolute_importance")
          .reset_index(drop=True)
    )

display(get_word_level_importance(result["shap_values"], top_n=10))


In [33]:

import numpy as np
import pandas as pd

DISPLAY_STOPWORDS = {
    "a", "an", "the", "to", "of", "in", "on", "at",
    "and", "or", "but", "is", "are", "was", "were",
    "with", "for", "by", "from", "according"
}

def get_word_importance(shap_values, top_n=10):
    values = shap_values[0]
    segments = values.data
    contributions = np.asarray(values.values).reshape(-1)

    # Merge SHAP segments into complete words.
    words = []
    current_word = ""
    current_score = 0.0

    for segment, score in zip(segments, contributions):
        segment = str(segment)

        # Empty boundary segments can mark the end of the text.
        if not segment.strip():
            if current_word:
                words.append((current_word, current_score))
                current_word = ""
                current_score = 0.0
            continue

        # Keep punctuation attached to the current word.
        piece = segment.strip()
        current_word += piece
        current_score += float(score)

        # A trailing space marks the end of a word.
        if segment[-1].isspace():
            words.append((current_word, current_score))
            current_word = ""
            current_score = 0.0

    # Flush any remaining word.
    if current_word:
        words.append((current_word, current_score))

    df = pd.DataFrame(words, columns=["token", "shap_value"])

    # Filter only for display; the model still receives the original text.
    df = df[
        df["token"].str.contains(r"[A-Za-z0-9]", regex=True)
    ].copy()

    df = df[
        ~df["token"].str.lower().str.strip(".,!?;:'\"()[]{}").isin(
            DISPLAY_STOPWORDS
        )
    ].copy()

    df["direction"] = np.where(
        df["shap_value"] > 0,
        "Pushes toward Biased",
        "Pushes away from Biased"
    )

    df["absolute_importance"] = df["shap_value"].abs()

    return (
        df.sort_values("absolute_importance", ascending=False)
          .head(top_n)
          .drop(columns="absolute_importance")
          .reset_index(drop=True)
    )


In [34]:

def analyze_news(text, top_n=10):
    text = text.strip()

    if not text:
        raise ValueError("Please enter some news text.")

    # Get class probabilities
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0].cpu().numpy()

    predicted_id = int(np.argmax(probabilities))
    predicted_label = model.config.id2label[predicted_id]

    # Explain the Biased-class probability
    values = explainer([text], max_evals=100)

    importance = get_word_importance(values, top_n=top_n)

    return {
        "text": text,
        "predicted_label": predicted_label,
        "probabilities": {
            model.config.id2label[i]: float(probabilities[i])
            for i in range(len(probabilities))
        },
        "word_importance": importance,
        "shap_values": values
    }


# Test the complete function
result = analyze_news(
    "The corrupt government shamelessly betrayed ordinary citizens "
    "with its disastrous policy."
)

print("Prediction:", result["predicted_label"])
print("\nProbabilities:")
for label, probability in result["probabilities"].items():
    print(f"{label}: {probability:.2%}")

print("\nMost influential tokens:")
display(result["word_importance"])


Prediction: Biased

Probabilities:
Non-biased: 0.30%
Biased: 99.70%

Most influential tokens:


,token,shap_value,direction
0,disastrous,0.200949,Pushes toward Biased
1,citizens,0.169178,Pushes toward Biased
2,corrupt,0.097835,Pushes toward Biased
3,shamelessly,0.088305,Pushes toward Biased
4,policy.,-0.076137,Pushes away from Biased
5,government,0.070570,Pushes toward Biased
6,betrayed,0.064367,Pushes toward Biased
7,its,0.036473,Pushes toward Biased
8,ordinary,0.020063,Pushes toward Biased


In [35]:

result = analyze_news(
    "The corrupt government shamelessly betrayed ordinary citizens "
    "with its disastrous policy."
)

display(result["word_importance"])


,token,shap_value,direction
0,disastrous,0.200949,Pushes toward Biased
1,citizens,0.169178,Pushes toward Biased
2,corrupt,0.097835,Pushes toward Biased
3,shamelessly,0.088305,Pushes toward Biased
4,policy.,-0.076137,Pushes away from Biased
5,government,0.070570,Pushes toward Biased
6,betrayed,0.064367,Pushes toward Biased
7,its,0.036473,Pushes toward Biased
8,ordinary,0.020063,Pushes toward Biased


In [39]:

test_samples = {
    "Strongly biased": (
        "The corrupt administration recklessly betrayed citizens "
        "with its disastrous decision."
    ),
    "Neutral reporting": (
        "The government announced a new policy on Tuesday, "
        "according to an official statement."
    ),
    "Ambiguous headline": (
        "Government policy sparks debate among citizens over "
        "the country's economic future."
    )
}

for category, text in test_samples.items():
    result = analyze_news(text, top_n=8)

    print("\n" + "=" * 65)
    print(f"TEST: {category}")
    print("=" * 65)
    print("Text:", result["text"])
    print("Prediction:", result["predicted_label"])

    for label, probability in result["probabilities"].items():
        print(f"{label}: {probability:.2%}")

    print("\nWord-level SHAP contributions:")
    display(result["word_importance"])



TEST: Strongly biased
Text: The corrupt administration recklessly betrayed citizens with its disastrous decision.
Prediction: Biased
Non-biased: 0.29%
Biased: 99.71%

Word-level SHAP contributions:


,token,shap_value,direction
0,disastrous,0.377339,Pushes toward Biased
1,corrupt,0.148315,Pushes toward Biased
2,recklessly,0.092009,Pushes toward Biased
3,betrayed,0.063760,Pushes toward Biased
4,its,0.040316,Pushes toward Biased
5,decision.,-0.039757,Pushes away from Biased
6,administration,0.030510,Pushes toward Biased
7,citizens,0.023992,Pushes toward Biased



TEST: Neutral reporting
Text: The government announced a new policy on Tuesday, according to an official statement.
Prediction: Non-biased
Non-biased: 99.26%
Biased: 0.74%

Word-level SHAP contributions:


,token,shap_value,direction
0,official,-0.117468,Pushes away from Biased
1,announced,-0.071680,Pushes away from Biased
2,statement.,0.036026,Pushes toward Biased
3,"Tuesday,",-0.030807,Pushes away from Biased
4,policy,-0.017686,Pushes away from Biased
5,government,-0.010076,Pushes away from Biased
6,new,-0.009329,Pushes away from Biased



TEST: Ambiguous headline
Text: Government policy sparks debate among citizens over the country's economic future.
Prediction: Non-biased
Non-biased: 99.39%
Biased: 0.61%

Word-level SHAP contributions:


,token,shap_value,direction
0,economic,-0.143306,Pushes away from Biased
1,among,-0.104425,Pushes away from Biased
2,policy,-0.089603,Pushes away from Biased
3,citizens,0.063325,Pushes toward Biased
4,debate,-0.058082,Pushes away from Biased
5,future.,-0.054852,Pushes away from Biased
6,sparks,0.045742,Pushes toward Biased
7,country's,-0.043865,Pushes away from Biased


In [40]:

import pandas as pd

evaluation_data = [
    # Neutral reporting
    ("Neutral", "The government announced a new education policy on Monday."),
    ("Neutral", "The central bank released its quarterly inflation report."),
    ("Neutral", "Officials confirmed that the bridge will reopen next week."),
    ("Neutral", "The health department published its annual statistics."),
    ("Neutral", "The committee will meet on Friday to discuss the proposal."),
    ("Neutral", "The company reported a 5 percent increase in annual revenue."),
    ("Neutral", "Police said the investigation remains ongoing."),
    ("Neutral", "The election commission announced the voting schedule."),
    ("Neutral", "The court postponed the hearing until next month."),
    ("Neutral", "The transport department released updated bus timetables."),

    # Emotionally loaded language
    ("Loaded language", "The corrupt administration pushed through a disastrous policy."),
    ("Loaded language", "The reckless decision betrayed hardworking citizens."),
    ("Loaded language", "The minister's shameful response angered residents."),
    ("Loaded language", "The company exploited vulnerable workers for enormous profits."),
    ("Loaded language", "The outrageous proposal threatens ordinary families."),
    ("Loaded language", "The irresponsible leadership ignored repeated warnings."),
    ("Loaded language", "The cruel policy devastated struggling communities."),
    ("Loaded language", "The politician launched a vicious attack on opponents."),
    ("Loaded language", "The disastrous plan exposed the government's incompetence."),
    ("Loaded language", "The greedy executives abandoned loyal employees."),

    # Opinionated framing
    ("Opinionated", "The mayor's bold reform is exactly what the city needs."),
    ("Opinionated", "The opposition's weak response failed to address the crisis."),
    ("Opinionated", "The proposed tax increase is an unfair burden on families."),
    ("Opinionated", "Supporters called the historic agreement a major victory."),
    ("Opinionated", "Critics described the new rules as unnecessary government control."),
    ("Opinionated", "The candidate offered a visionary plan for economic renewal."),
    ("Opinionated", "The administration's latest move raises serious questions."),
    ("Opinionated", "The policy is a sensible step toward long-term stability."),
    ("Opinionated", "The proposal is a misguided attempt to solve a complex problem."),
    ("Opinionated", "The reform could reshape the country's economic future."),
]

eval_df = pd.DataFrame(
    evaluation_data,
    columns=["category", "text"]
)

eval_df.insert(0, "example_id", range(1, len(eval_df) + 1))

display(eval_df.head(10))
print("Total examples:", len(eval_df))
print("\nExamples per category:")
display(eval_df["category"].value_counts().to_frame("count"))


,example_id,category,text
0,1,Neutral,The government announced a new education polic...
1,2,Neutral,The central bank released its quarterly inflat...
2,3,Neutral,Officials confirmed that the bridge will reope...
3,4,Neutral,The health department published its annual sta...
4,5,Neutral,The committee will meet on Friday to discuss t...
5,6,Neutral,The company reported a 5 percent increase in a...
6,7,Neutral,Police said the investigation remains ongoing.
7,8,Neutral,The election commission announced the voting s...
8,9,Neutral,The court postponed the hearing until next month.
9,10,Neutral,The transport department released updated bus ...


Total examples: 30

Examples per category:


,count
category,
Neutral,10
Loaded language,10
Opinionated,10


In [41]:

results = []

for _, row in eval_df.iterrows():
    prediction = analyze_news(row["text"], top_n=5)

    results.append({
        "example_id": row["example_id"],
        "category": row["category"],
        "text": row["text"],
        "predicted_label": prediction["predicted_label"],
        "non_biased_probability": prediction["probabilities"]["Non-biased"],
        "biased_probability": prediction["probabilities"]["Biased"],
        "top_words": ", ".join(
            prediction["word_importance"]["token"].tolist()
        )
    })

results_df = pd.DataFrame(results)

display(
    results_df[
        [
            "example_id",
            "category",
            "predicted_label",
            "biased_probability",
            "top_words"
        ]
    ]
)


,example_id,category,predicted_label,biased_probability,top_words
0,1,Neutral,Non-biased,0.006857,"announced, government, Monday., policy, education"
1,2,Neutral,Non-biased,0.008823,"released, bank, central, report., quarterly"
2,3,Neutral,Non-biased,0.006215,"confirmed, that, Officials, bridge, week."
3,4,Neutral,Non-biased,0.006837,"department, published, annual, health, statist..."
4,5,Neutral,Non-biased,0.007190,"discuss, proposal., committee, Friday, will"
5,6,Neutral,Non-biased,0.006018,"company, revenue., annual, percent, reported"
6,7,Neutral,Non-biased,0.007506,"Police, ongoing., said, remains, investigation"
7,8,Neutral,Non-biased,0.005946,"announced, election, commission, schedule., vo..."
8,9,Neutral,Non-biased,0.006650,"court, hearing, month., postponed, until"
9,10,Neutral,Non-biased,0.007175,"timetables., department, transport, updated, r..."


In [42]:

summary = (
    results_df
    .groupby(["category", "predicted_label"])
    .size()
    .unstack(fill_value=0)
)

display(summary)

print("Mean biased probability by category:")
display(
    results_df.groupby("category")["biased_probability"]
    .mean()
    .sort_values(ascending=False)
    .to_frame("mean_biased_probability")
)


predicted_label,Biased,Non-biased
category,,
Loaded language,10,0
Neutral,0,10
Opinionated,2,8


Mean biased probability by category:


,mean_biased_probability
category,
Loaded language,0.993159
Opinionated,0.222488
Neutral,0.006922


In [43]:

eval_df.to_csv("news_bias_evaluation_examples.csv", index=False)
results_df.to_csv("news_bias_evaluation_results.csv", index=False)

print("Saved both CSV files.")


Saved both CSV files.


In [44]:

opinionated_results = results_df[
    results_df["category"] == "Opinionated"
].copy()

opinionated_results["biased_probability"] = (
    opinionated_results["biased_probability"] * 100
).round(2)

display(
    opinionated_results[
        [
            "example_id",
            "text",
            "predicted_label",
            "biased_probability",
            "top_words"
        ]
    ].sort_values("biased_probability", ascending=False)
)


,example_id,text,predicted_label,biased_probability,top_words
28,29,The proposal is a misguided attempt to solve a...,Biased,99.62,"misguided, solve, proposal, problem., attempt"
21,22,The opposition's weak response failed to addre...,Biased,97.37,"weak, failed, address, crisis., opposition's"
20,21,The mayor's bold reform is exactly what the ci...,Non-biased,14.40,"city, exactly, needs., mayor's, what"
25,26,The candidate offered a visionary plan for eco...,Non-biased,3.78,"candidate, offered, plan, visionary, renewal."
26,27,The administration's latest move raises seriou...,Non-biased,2.92,"administration's, serious, questions., raises,..."
27,28,The policy is a sensible step toward long-term...,Non-biased,1.31,"step, stability., sensible, policy, long-term"
22,23,The proposed tax increase is an unfair burden ...,Non-biased,1.08,"increase, unfair, tax, proposed, burden"
24,25,Critics described the new rules as unnecessary...,Non-biased,0.68,"Critics, control., described, unnecessary, rules"
29,30,The reform could reshape the country's economi...,Non-biased,0.68,"reform, economic, could, reshape, future."
23,24,Supporters called the historic agreement a maj...,Non-biased,0.65,"Supporters, called, major, victory., agreement"


In [45]:

examples_to_check = {
    23: "The proposed tax increase is an unfair burden on families.",
    25: "Critics described the new rules as unnecessary government control."
}

for example_id, text in examples_to_check.items():
    result = analyze_news(text, top_n=15)

    print("\n" + "=" * 70)
    print(f"Example {example_id}")
    print("Text:", text)
    print("Prediction:", result["predicted_label"])
    print(
        f"Biased probability: "
        f"{result['probabilities']['Biased']:.2%}"
    )

    print("\nSHAP word contributions:")
    display(result["word_importance"])



Example 23
Text: The proposed tax increase is an unfair burden on families.
Prediction: Non-biased
Biased probability: 1.08%

SHAP word contributions:


,token,shap_value,direction
0,increase,-0.117493,Pushes away from Biased
1,unfair,0.106393,Pushes toward Biased
2,tax,0.076102,Pushes toward Biased
3,proposed,-0.071762,Pushes away from Biased
4,burden,-0.005469,Pushes away from Biased
5,families.,0.002497,Pushes toward Biased



Example 25
Text: Critics described the new rules as unnecessary government control.
Prediction: Non-biased
Biased probability: 0.68%

SHAP word contributions:


,token,shap_value,direction
0,Critics,-0.321791,Pushes away from Biased
1,control.,0.304886,Pushes toward Biased
2,described,-0.245402,Pushes away from Biased
3,unnecessary,0.192431,Pushes toward Biased
4,rules,-0.112756,Pushes away from Biased
5,as,-0.085792,Pushes away from Biased
6,new,0.047081,Pushes toward Biased
7,government,-0.046622,Pushes away from Biased
